# IMPORTS

In [21]:
import requests

import json
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_selection import mutual_info_classif
from scipy.stats import pointbiserialr
import os
import numpy as np
import seaborn as sns
import datetime

# Data read

In [4]:
prices  = pd.read_parquet("../data/prices_clean/", engine="pyarrow")

# Feature calculus for finance

## Log return

In [9]:
prices = prices.sort_values(["ticker", "date"])

In [10]:
#lag es para mirar n filas anteriores
#definir en la ventana como se va a ordenar y dividir el dataset
prices["log_return"] = (
    np.log(
        prices["close"] /
        prices.groupby("ticker")["close"].shift(1)
    )
)

## RSI

In [13]:
prices["difference"] = prices.groupby("ticker")["close"].diff(1)
prices["gain"] = prices["difference"].where(prices["difference"] > 0, 0)
prices["loss"] = prices["difference"].where(prices["difference"] < 0, 0).abs()
window=14
prices["avg_gains"] = prices.groupby("ticker")["gain"].rolling(window, min_periods=window).mean().reset_index(level=0, drop=True)
prices["avg_loses"] = prices.groupby("ticker")["loss"].rolling(window, min_periods=window).mean().reset_index(level=0, drop=True)
prices["rsi"] = 100 - (100 / (1 + prices["avg_gains"] / prices["avg_loses"]))
prices = prices.drop(columns=["difference", "gain", "loss", "avg_gains", "avg_loses"])

## MACD

In [14]:
prices["ma_12"] = (
    prices.groupby("ticker")["close"]
    .rolling(window=12, min_periods=12)
    .mean()
    .reset_index(level=0, drop=True)
)

prices["ma_26"] = (
    prices.groupby("ticker")["close"]
    .rolling(window=26, min_periods=26)
    .mean()
    .reset_index(level=0, drop=True)
)

prices["macd"] = prices["ma_12"] - prices["ma_26"]
prices.loc[prices["ma_12"].isna() | prices["ma_26"].isna(), "macd"] = np.nan


## Bollinger Bands

In [15]:
window = 20
prices["std_20"] = (
    prices.groupby("ticker")["close"]
    .rolling(window, min_periods=window)
    .std()
    .reset_index(level=0, drop=True)
)

prices["bb_mid"] = (
    prices.groupby("ticker")["close"]
    .rolling(window, min_periods=window)
    .mean()
    .reset_index(level=0, drop=True)
)

prices["bb_upper"] = prices["bb_mid"] + 2 * prices["std_20"]
prices["bb_lower"] = prices["bb_mid"] - 2 * prices["std_20"]

prices["bb_position"] = (
    (prices["close"] - prices["bb_lower"]) /
    (prices["bb_upper"] - prices["bb_lower"])
)

prices.loc[prices["bb_mid"].isna() | prices["bb_upper"].isna() | prices["bb_lower"].isna(), "bb_position"] = np.nan



## Historic volatility

In [16]:
window = 20
prices["volatility"] = (
    prices.groupby("ticker")["log_return"]
    .rolling(window, min_periods=window)
    .std()
    .reset_index(level=0, drop=True)
)

## Volume ratio

In [17]:
window = 10

prices["vol_ma_10"] = (
    prices.groupby("ticker")["volume"]
    .rolling(window, min_periods=window)
    .mean()
    .reset_index(level=0, drop=True)
)

prices["vol_ratio"] = prices["volume"] / prices["vol_ma_10"]
prices.loc[prices["vol_ma_10"].isna(), "vol_ratio"] = np.nan
prices = prices.drop(columns=["vol_ma_10"])

## Target

In [0]:
prices = prices.withColumn('difference',(col("close")-lag("close",1).over(w_1)))
prices = prices.withColumn('target',when(col('difference')>0,1).otherwise(0))
prices = prices.drop('difference')
prices.select('ticker','date','close','target').show(30)


In [18]:
prices = prices.sort_values(["ticker", "date"]).copy()
prices["previous"] = prices.groupby("ticker")["close"].shift(1)
prices["target"] = (prices["close"] > prices["previous"]).astype(int)

# Load

In [20]:
feature_store_technical = prices[[
    "ticker", "date", "year", "week",
    "close", "volume",
    "log_return", "rsi", "macd", "bb_position", 
    "volatility", "vol_ratio",
    "target"
]]



In [22]:
path = "../data/features/"

for (ticker, year), group in feature_store_technical.groupby(["ticker", "year"]):
    dir_path = f"{path}/ticker={ticker}/year={year}"
    os.makedirs(dir_path, exist_ok=True)
    
    # 🔥 quitar columnas de partición
    group = group.drop(columns=["ticker", "year"])
    
    group.to_parquet(f"{dir_path}/data.parquet", index=False)

# Feature exploration

In [0]:
prices.select("ticker").distinct().show()

In [0]:
nvda = prices.filter(prices['ticker'] == 'NVDA')

In [0]:
nvda.select([count(when(col(c).isNull(), c)).alias(c) for c in nvda.columns]).show()

los nulos tienen sentido porqeu son valores que de penden de n días anteriores, por lo que si ni tienen esos n días, no
calculan y dan un nulo


In [0]:
nvda.printSchema()

In [0]:
type(nvda)

In [0]:
nvda_pd = nvda.toPandas()
nvda_pd=nvda_pd[nvda_pd['year']>=2024]
plt.figure(figsize=(15,10))
plt.plot(nvda_pd['date'],nvda_pd['close'],label='close')
plt.plot(nvda_pd['date'],nvda_pd['bb_upper'],label='bb_upper')
plt.plot(nvda_pd['date'],nvda_pd['bb_lower'],label='bb_lower')
plt.plot(nvda_pd['date'],nvda_pd['bb_mid'])
plt.plot(nvda_pd['date'],nvda_pd['volatility'],label='volatility')
plt.plot(nvda_pd['date'],nvda_pd['rsi'],label='rsi')
plt.legend()
plt.show()

In [0]:
nvda_target_count = nvda_pd.groupby('target')['date'].count().reset_index()
plt.pie(nvda_target_count['date'],labels=nvda_target_count['target'].unique())

# Correlaciones y relación entre variables

In [0]:
numeric_cols = [c.name for c in feature_store_technical.schema if isinstance(c.dataType,NumericType) and c.name != 'target']

In [0]:
assembler = VectorAssembler(
    inputCols=numeric_cols+['target'],
    outputCol='features',
    handleInvalid='skip')#saltar los nulos
df_vector = assembler.transform(feature_store_technical).select('features')


In [0]:
corr_mat = Correlation.corr(df_vector,'features',method='pearson')
corr_mat = corr_mat.collect()[0]['pearson(features)'].toArray()
display(corr_mat[-1])
plt.figure(figsize=(15,10))




In [0]:
sns.heatmap(np.asarray(corr_mat[-1]).reshape(11,1),yticklabels=numeric_cols, annot=True)

In [0]:
# Mutual Information
f_pd = feature_store_technical.toPandas().dropna()
mi = mutual_info_classif(f_pd[numeric_cols], f_pd["target"])
for col, score in zip(numeric_cols, mi):
    print(f"MI  {col}: {score:.4f}")

In [0]:
for col in numeric_cols:
    corr, pval = pointbiserialr(f_pd[col], f_pd["target"])
    print(f"PB  {col}: {corr:.4f} (p={pval:.4f})")